In [ ]:
!pip install pyngrok langchain-community streamlit wikipedia
!pip install -U langchain-community

## Build a Mini Chatbot Webapp:
We'll create a Streamlit webapp that generates YouTube video titles and scripts using LangChain and the locally loaded `google/flan-t5-large` model on CUDA.
### Setup for Colab:

1. Save the code as `app.py` using `%%writefile app.py`.
2. Run Streamlit with `!streamlit run app.py --server.port 8501 &`.
3. Use `ngrok` to access the app:

### Step 1: Load the Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain.llms import HuggingFacePipeline

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Create a pipeline for LangChain
pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=512,
    temperature=0.6,
    device=0 if torch.cuda.is_available() else -1
)
llm = HuggingFacePipeline(pipeline=pipe)

## Step 2: Streamlit and Prompt Templates

In [ ]:
import streamlit as st
st.title('Genify Bot')
input_text = st.text_input('Enter Your Text:')

from langchain.prompts import PromptTemplate

title_template = PromptTemplate(
    input_variables=['concept'],
    template='Give me a youtube video title about {concept}'
)

script_template = PromptTemplate(
    input_variables=['title', 'wikipedia_research'],
    template='''Give me an attractive youtube video script based on the title {title} while making use of the information and knowledge obtained from the Wikipedia research:{wikipedia_research}'''
)

## Step 3: Memory and Chains

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import LLMChain

memoryT = ConversationBufferMemory(input_key='concept', memory_key='chat_history')
memoryS = ConversationBufferMemory(input_key='title', memory_key='chat_history')

chainT = LLMChain(llm=llm, prompt=title_template, verbose=True, output_key='title', memory=memoryT)
chainS = LLMChain(llm=llm, prompt=script_template, verbose=True, output_key='script', memory=memoryS)

## Step 4: Wikipedia Integration and Output

In [ ]:
from langchain.utilities import WikipediaAPIWrapper

wikipedia = WikipediaAPIWrapper()

if input_text:
    title = chainT.run(input_text)
    wikipedia_research = wikipedia.run(input_text)
    script = chainS.run(title=title, wikipedia_research=wikipedia_research)
    st.write(title)
    st.write(script)
    with st.expander('Wikipedia-based exploration:'):
        st.info(wikipedia_research)

## Step 5: Put it all in One File (App.py)

In [ ]:
%%writefile app.py
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain.llms import HuggingFacePipeline
import streamlit as st

# Set the title using Streamlit
st.title('Genify Bot')
input_text = st.text_input('Enter Your Text:')

from langchain.prompts import PromptTemplate
# Setting up the prompt templates
title_template = PromptTemplate(
    input_variables=['concept'],
    template='Give me a youtube video title about {concept}'
)

script_template = PromptTemplate(
    input_variables=['title', 'wikipedia_research'],
    template='''Give me an attractive youtube video script based on the title {title} while making use of the information and knowledge obtained from the Wikipedia research:{wikipedia_research}'''
)

from langchain.memory import ConversationBufferMemory
# Store conversation history
memoryT = ConversationBufferMemory(input_key='concept', memory_key='chat_history')
memoryS = ConversationBufferMemory(input_key='title', memory_key='chat_history')

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Create a pipeline for LangChain
pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=512,
    temperature=0.6,
    device=0 if torch.cuda.is_available() else -1
)
llm = HuggingFacePipeline(pipeline=pipe)

from langchain.chains import LLMChain
chainT = LLMChain(llm=llm, prompt=title_template, verbose=True, output_key='title', memory=memoryT)
chainS = LLMChain(llm=llm, prompt=script_template, verbose=True, output_key='script', memory=memoryS)

from langchain.utilities import WikipediaAPIWrapper
wikipedia = WikipediaAPIWrapper()

# Display the output if the user gives an input
if input_text:
    title = chainT.run(input_text)
    wikipedia_research = wikipedia.run(input_text)
    script = chainS.run(title=title, wikipedia_research=wikipedia_research)
    st.write(title)
    st.write(script)
    with st.expander('Wikipedia-based exploration:'):
        st.info(wikipedia_research)

In [ ]:
from pyngrok import ngrok

# Replace "YOUR_AUTHTOKEN" with your actual ngrok authtoken
# ngrok.set_auth_token("YOUR_AUTHTOKEN")
ngrok.set_auth_token("YOUR_AUTH_TOKEN")

public_url = ngrok.connect(8501)
print(public_url)

!streamlit run app.py --server.port 8501 &
public_url = ngrok.connect(8501)
print(public_url)